In [4]:
import os, shutil, pathlib

SRC_IN  = "/kaggle/input/datasets/omarcodeplay/gm-dataset/src"
SRC_OUT = "/kaggle/working/src"

# 1. Verify input mount exists
if not os.path.isdir(SRC_IN):
    # Fallback: search common dataset paths
    import glob
    candidates = glob.glob("/kaggle/input/**/src", recursive=True)
    if candidates:
        SRC_IN = candidates[0]
        print(f"found src at: {SRC_IN}")
    else:
        raise SystemExit(
            f"Source not found at {SRC_IN}.\n"
            f"Check 'Add Data' in the Kaggle notebook sidebar."
        )

# 2. Copy everything
os.makedirs(SRC_OUT, exist_ok=True)
copied = []
for f in pathlib.Path(SRC_IN).glob("*.py"):
    dst = pathlib.Path(SRC_OUT) / f.name
    shutil.copy(f, dst)
    copied.append(f.name)

# 3. Also copy config.json if present (train.py looks for it via sync)
cfg = pathlib.Path(SRC_IN).parent / "config.json"
if cfg.exists():
    shutil.copy(cfg, "/kaggle/working/config.json")
    print(f"copied config.json")

print(f"✅ Copied {len(copied)} files to {SRC_OUT}:")
for f in sorted(copied):
    print(f"  {f}")

copied config.json
✅ Copied 11 files to /kaggle/working/src:
  bench.py
  build_data.py
  download_data.py
  encode.py
  engine.py
  gauntlet.py
  model.py
  selftest.py
  sign_check.py
  sync.py
  train.py


In [5]:
%%bash
pip install -q --upgrade python-chess huggingface_hub pandas numpy zstandard orjson

In [ ]:
import os, sys
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USER"] = "OmarCodePlay"
for k in list(sys.modules):
    if k == "sync" or k.startswith("sync."):
        del sys.modules[k]
sys.path.insert(0, "/kaggle/working/src")
import sync

import pathlib, shutil
SRC_IN = "/kaggle/input/datasets/omarcodeplay/gm-dataset/src"
SRC = pathlib.Path("/kaggle/working/src")
SRC.mkdir(parents=True, exist_ok=True)
if os.path.isdir(SRC_IN):
    for f in pathlib.Path(SRC_IN).glob("*.py"):
        shutil.copy(f, SRC / f.name)
    print(f"✅ copied {len(list(SRC.glob('*.py')))} src files")

# Patch sync.py env precedence
sp = SRC / "sync.py"
if sp.exists():
    s = sp.read_text()
    old = '''try:
    from kaggle_secrets import UserSecretsClient
    TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
except Exception:
    TOKEN = os.environ.get("HF_TOKEN")'''
    new = '''TOKEN = os.environ.get("HF_TOKEN")
if not TOKEN:
    try:
        from kaggle_secrets import UserSecretsClient
        TOKEN = UserSecretsClient().get_secret("HF_TOKEN")
    except Exception:
        TOKEN = None
if not TOKEN:
    raise RuntimeError("HF_TOKEN not found")'''
    if old in s:
        sp.write_text(s.replace(old, new))
    for k in list(sys.modules):
        if k == "sync" or k.startswith("sync."):
            del sys.modules[k]
    import sync

# Download 2024 games (your fresh merge)
os.makedirs("/kaggle/working/data", exist_ok=True)
p = sync.pull_data("Hikaru-Moves-2024.csv", "/kaggle/working/data")
print(f"✅ Hikaru-Moves-2024.csv: {os.path.getsize(p)/1e6:.1f} MB")

import pandas as pd
df = pd.read_csv(p, low_memory=False)
print(f"   rows: {len(df):,}")
print(f"   result dist:")
print(df["result"].value_counts().to_string())

In [ ]:
import os, sys, glob
sys.path.insert(0, "/kaggle/working/src")
import numpy as np, pandas as pd, chess
from encode import pack_board, move_index, save_npz

# Download parquet files if missing
parquet_files = sorted(glob.glob("/kaggle/working/default/train/*.parquet"))
if len(parquet_files) < 3:
    print("Downloading Lichess puzzles...")
    from huggingface_hub import hf_hub_download
    for i in ["0000", "0001", "0002"]:
        p = hf_hub_download(
            repo_id="Lichess/chess-puzzles",
            filename=f"default/train/{i}.parquet",
            repo_type="dataset", revision="refs/convert/parquet",
            token=os.environ["HF_TOKEN"], local_dir="/kaggle/working")
        print(f"  {i}: {os.path.getsize(p)/1e6:.1f} MB")
    parquet_files = sorted(glob.glob("/kaggle/working/default/train/*.parquet"))

print(f"Loading {len(parquet_files)} parquet files...")
parts = [pd.read_parquet(f) for f in parquet_files]
df = pd.concat(parts, ignore_index=True)
print(f"Total puzzles: {len(df):,}")

# Theme token helper (parquet may store as list or string)
def theme_tokens(themes):
    if themes is None: return []
    if isinstance(themes, str): return themes.split()
    try: return [str(t) for t in themes]
    except: return str(themes).split()

# Filter to mate-only puzzles
MATE_THEMES = {"mateIn1", "mateIn2", "mateIn3", "mateIn4"}
def is_mate_puzzle(themes):
    return any(t in MATE_THEMES for t in theme_tokens(themes))

df["_is_mate"] = df["Themes"].apply(is_mate_puzzle)
mate_df = df[df["_is_mate"]].copy()
print(f"Mate puzzles (all): {len(mate_df):,}")

# Standard filters
mate_df = mate_df[(mate_df["Rating"] >= 1200) & (mate_df["Rating"] <= 2600)]
print(f"After rating:       {len(mate_df):,}")
mate_df = mate_df[mate_df["RatingDeviation"] <= 100]
print(f"After max RD:       {len(mate_df):,}")
mate_df = mate_df[mate_df["Popularity"].fillna(-100) >= 80]
print(f"After popularity:   {len(mate_df):,}")
mate_df = mate_df[mate_df["NbPlays"].fillna(0) >= 200]
print(f"After plays:        {len(mate_df):,}")

# Theme → value mapping (mate = +1.0 always)
def mate_value(themes):
    t = set(theme_tokens(themes))
    if "mateIn1" in t: return 1.0
    if "mateIn2" in t: return 1.0
    if "mateIn3" in t: return 1.0
    if "mateIn4" in t: return 1.0
    return 1.0   # all mate → +1

# Build samples
samples = []
seen_fens = set()
skipped = 0
for r in mate_df.to_dict("records"):
    fen = str(r.get("FEN", "") or "")
    moves_str = str(r.get("Moves", "") or "")
    if not fen or not moves_str: continue

    try:
        board = chess.Board(fen)
    except: continue

    moves = moves_str.split()
    if len(moves) < 2: continue

    try:
        setup = chess.Move.from_uci(moves[0])
        if setup not in board.legal_moves: continue
        board.push(setup)
    except: continue

    # First solver move = mating move
    try:
        sol = chess.Move.from_uci(moves[1])
        if sol not in board.legal_moves: continue
    except: continue

    fen_before = board.fen()
    if fen_before in seen_fens: continue
    seen_fens.add(fen_before)

    black = board.turn == chess.BLACK
    try:
        bb, meta = pack_board(board)
    except:
        skipped += 1
        continue

    v = mate_value(r.get("Themes", ""))
    samples.append((
        bb, meta,
        np.array([move_index(sol, black), -1, -1, -1], dtype=np.int16),
        np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float16),
        float(v),
    ))

print(f"\nMate samples: {len(samples):,}  (skipped {skipped})")

# Value distribution (should be 100% +1.0)
if samples:
    vals = np.array([s[4] for s in samples])
    print(f"Value: min={vals.min():.2f} max={vals.max():.2f} mean={vals.mean():.2f}")

# Save
os.makedirs("/kaggle/working/shards", exist_ok=True)
SHARD_SIZE = 300_000
mate_shard_names = []
for i in range(0, len(samples), SHARD_SIZE):
    chunk = samples[i:i + SHARD_SIZE]
    fn = f"mate_{i // SHARD_SIZE:04d}.npz"
    save_npz(chunk, f"/kaggle/working/shards/{fn}")
    print(f"  saved {fn}  ({len(chunk):,} rows)")
    mate_shard_names.append(fn)

In [ ]:
import os, sys, numpy as np, pandas as pd, chess, math
sys.path.insert(0, "/kaggle/working/src")
from encode import pack_board, move_index, save_npz

df = pd.read_csv("/kaggle/working/data/Hikaru-Moves-2024.csv", low_memory=False)
print(f"Total 2024: {len(df):,}")

draws = df[df["result"] == "1/2-1/2"].copy()
print(f"Draw positions: {len(draws):,}")

draws["eval_depth_before"] = pd.to_numeric(draws["eval_depth_before"], errors="coerce")
draws = draws[draws["eval_depth_before"] >= 12]
print(f"With depth≥12:  {len(draws):,}")

def cp_to_v(cp):
    return 2.0 / (1.0 + math.exp(-0.00368208 * cp)) - 1.0

def parse_mate(x):
    if pd.isna(x): return None
    try: f = float(x)
    except: return None
    if f == 0: return None
    return 1.0 if f > 0 else -1.0

draws["eval_cp_before"] = pd.to_numeric(draws["eval_cp_before"], errors="coerce")
draws["eval_mate_before"] = pd.to_numeric(draws["eval_mate_before"], errors="coerce")

samples = []
seen_fens = set()
skipped = 0
for _, r in draws.iterrows():
    try:
        fen = r["fen"]
        if fen in seen_fens: continue
        seen_fens.add(fen)

        b = chess.Board(fen)
        m = chess.Move.from_uci(str(r["move_uci"]))
        if m not in b.legal_moves:
            skipped += 1
            continue
        black = b.turn == chess.BLACK

        mate_v = parse_mate(r["eval_mate_before"])
        if mate_v is not None:
            v = mate_v
        else:
            cp = r["eval_cp_before"]
            if pd.isna(cp):
                skipped += 1
                continue
            v = cp_to_v(float(cp))

        if black: v = -v

        bb, meta = pack_board(b)
        samples.append((
            bb, meta,
            np.array([move_index(m, black), -1, -1, -1], dtype=np.int16),
            np.array([1.0, 0.0, 0.0, 0.0], dtype=np.float16),
            float(v),
        ))
    except:
        skipped += 1
        continue

print(f"Draw samples: {len(samples):,}  (skipped {skipped})")

vals = np.array([s[4] for s in samples])
print(f"Value: min={vals.min():.3f} max={vals.max():.3f} mean={vals.mean():.3f}")
print(f"  |v| < 0.15 (truly drawn):  {(np.abs(vals) < 0.15).mean()*100:5.1f}%")
print(f"  |v| < 0.30 (near-draw):    {(np.abs(vals) < 0.30).mean()*100:5.1f}%")

os.makedirs("/kaggle/working/shards", exist_ok=True)
draw_shard_names = []
SHARD_SIZE = 200_000
for i in range(0, len(samples), SHARD_SIZE):
    chunk = samples[i:i + SHARD_SIZE]
    fn = f"draws_{i // SHARD_SIZE:04d}.npz"
    save_npz(chunk, f"/kaggle/working/shards/{fn}")
    print(f"  saved {fn}  ({len(chunk):,} rows)")
    draw_shard_names.append(fn)

In [ ]:
import os, sys, json
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USER"] = "OmarCodePlay"
for k in list(sys.modules):
    if k == "sync" or k.startswith("sync."):
        del sys.modules[k]
sys.path.insert(0, "/kaggle/working/src")
import sync

# Push all new shards
all_new = mate_shard_names + draw_shard_names + ["resign_0000.npz"]
for fn in all_new:
    if fn == "resign_0000.npz":
        # already pushed earlier
        continue
    sync.push_data(f"/kaggle/working/shards/{fn}", fn)
    print(f"✅ pushed {fn}")

# Load current index
idx = json.load(open(sync.pull_data("index.json", "/kaggle/working/shards")))

# Rebuild train list to include everything
new_train = []

# Games (2 shards from previous pipeline)
for s in ["train_0000.npz", "train_0001.npz"]:
    if s not in new_train:
        new_train.append(s)

# Regular puzzles (maybe keep 1 shard if present)
for s in idx["train"]:
    if s.startswith("puzz_") and s not in new_train:
        new_train.append(s)

# Resignation samples
if "resign_0000.npz" not in new_train:
    new_train.append("resign_0000.npz")

# NEW: Mate puzzles (the fix for checkmate)
for s in mate_shard_names:
    if s not in new_train:
        new_train.append(s)

# NEW: Draw positions
for s in draw_shard_names:
    if s not in new_train:
        new_train.append(s)

idx["train"] = new_train
with open("/tmp/index.json", "w") as f:
    json.dump(idx, f)
sync.push_data("/tmp/index.json", "index.json")

print(f"\n✅ index.json updated")
print(f"  Total train shards: {len(idx['train'])}")
for s in idx["train"]:
    print(f"    {s}")

In [ ]:
import pathlib
p = pathlib.Path("/kaggle/working/src/train.py")
src = p.read_text()

src = src.replace("map_location=local_rank", 'map_location=f"cuda:{local_rank}"')
src = src.replace('run="base7",', 'run="v5",')
src = src.replace('run="base6",', 'run="v5",')
src = src.replace('run="base5",', 'run="v5",')
src = src.replace('init_from="engine.pt",', 'init_from="engine.pt",')  # start from v3
src = src.replace('lr=3e-5,', 'lr=4e-5,')       # gentle fine-tune
src = src.replace('lr=5e-5,', 'lr=4e-5,')
src = src.replace('v_w=0.8,', 'v_w=0.7,')
src = src.replace('v_w=0.5,', 'v_w=0.7,')
src = src.replace('max_steps=12000,', 'max_steps=15000,')  # more data = more steps
src = src.replace('max_steps=10000,', 'max_steps=15000,')
src = src.replace('decay_start=9000,', 'decay_start=11000,')
src = src.replace('decay_start=7500,', 'decay_start=11000,')
src = src.replace('decay_steps=3000,', 'decay_steps=4000,')
src = src.replace('decay_steps=2500,', 'decay_steps=4000,')

p.write_text(src)
print("✅ train.py configured for v5")

!grep -nE "run=|init_from=|lr=|v_w=|max_steps=|decay_start=" /kaggle/working/src/train.py | head -8

In [ ]:
!grep -n "has_policy" /kaggle/working/src/train.py

In [ ]:
import os, sys
from kaggle_secrets import UserSecretsClient
os.environ["HF_TOKEN"] = UserSecretsClient().get_secret("HF_TOKEN")
os.environ["HF_USER"] = "OmarCodePlay"
for k in list(sys.modules):
    if k == "sync" or k.startswith("sync."):
        del sys.modules[k]

!cd /kaggle/working && torchrun --standalone --nproc_per_node=2 src/train.py 2>&1 | tee -a /kaggle/working/train_v5.log

In [ ]:
import chess
from engine import Engine
eng = Engine("/kaggle/working/ckpt/best_v5.pt")

# Mate-in-2 test position (back rank mate)
fen = "6k1/5ppp/8/8/8/8/5PPP/R5K1 w - - 0 1"   # Ra8+ → mate
board = chess.Board(fen)
move, root = eng.search(board, float("inf"), float("inf"), 500, None)
print(f"v5 plays: {move} (SAN: {board.san(move)})")
print(f"expected: Ra8+ (checkmate on next move)")

# Compare with base3
eng3 = Engine("/kaggle/working/ckpt/engine.pt")
move3, _ = eng3.search(board, float("inf"), float("inf"), 500, None)
print(f"base3 plays: {move3} (SAN: {board.san(move3)})")